In [3]:
import pathlib
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from tqdm.auto import tqdm

In [ ]:
JOBS_DIR = pathlib.Path("rollout_jobs")
SUCCESS_DIR = pathlib.Path("rollout_successes")
DATASETS_ROOT = pathlib.Path("/vision/group/vid2room/rollouts")

jobs = sorted(JOBS_DIR.glob("*.csv"))
records = []
for job_file in tqdm(jobs):
    success_file = SUCCESS_DIR / job_file.with_suffix(".success").relative_to(JOBS_DIR)
    if not success_file.exists():
        print(f"Success file {success_file} does not exist")
        continue

    dataset_name, split, scene, rollout_count = job_file.read_text().strip().split(",")
    lr_dataset_path = DATASETS_ROOT / f"{dataset_name}-{split}" / success_file.read_text().strip()
    try:
      lr_dataset = LeRobotDataset(root=lr_dataset_path, repo_id=lr_dataset_path.name)
      if lr_dataset.num_episodes != int(rollout_count):
          print(f"Rollout count mismatch for {job_file}")

      records.append({
          "dataset_name": dataset_name,
          "split": split,
          "scene": scene,
          "num_episodes": lr_dataset.num_episodes,
          "lr_dataset_path": lr_dataset_path,
      })
      print(f"Successfully processed {job_file}")
    except Exception as e:
      print(f"Error processing {job_file}")
      continue

df = pd.DataFrame(records)


  0%|          | 0/329 [00:00<?, ?it/s]

Error processing rollout_jobs/0.csv
Successfully processed rollout_jobs/1.csv
Successfully processed rollout_jobs/10.csv
Error processing rollout_jobs/100.csv
Error processing rollout_jobs/101.csv
Successfully processed rollout_jobs/102.csv
Successfully processed rollout_jobs/103.csv
Error processing rollout_jobs/104.csv
Successfully processed rollout_jobs/105.csv
Successfully processed rollout_jobs/106.csv
Error processing rollout_jobs/107.csv
Successfully processed rollout_jobs/108.csv
Error processing rollout_jobs/109.csv
Successfully processed rollout_jobs/11.csv
Successfully processed rollout_jobs/110.csv
Successfully processed rollout_jobs/111.csv
Error processing rollout_jobs/112.csv
Error processing rollout_jobs/113.csv
Rollout count mismatch for rollout_jobs/114.csv
Successfully processed rollout_jobs/114.csv
Error processing rollout_jobs/115.csv
Successfully processed rollout_jobs/116.csv
Error processing rollout_jobs/117.csv
Error processing rollout_jobs/118.csv
Successfully

In [11]:
len(df)

191

In [16]:
# Is every scene unique in the list?
assert df["scene"].nunique() == len(df)

In [ ]:
for min_episodes in [1, 2, 5, 10]:
    df_min = df[(df["num_episodes"] >= min_episodes) & (df["split"] == "train")]
    print(f"Number of datasets with at least {min_episodes} episodes: {len(df_min)}")
    print(df_min.groupby("dataset_name").size())
    print()

Number of datasets with at least 1 episodes: 164
dataset_name
behavior-1k-assets    10
spoc                  81
vid2room              73
dtype: int64

Number of datasets with at least 2 episodes: 150
dataset_name
behavior-1k-assets    10
spoc                  72
vid2room              68
dtype: int64

Number of datasets with at least 5 episodes: 138
dataset_name
behavior-1k-assets    10
spoc                  67
vid2room              61
dtype: int64

Number of datasets with at least 10 episodes: 130
dataset_name
behavior-1k-assets    10
spoc                  61
vid2room              59
dtype: int64



In [17]:
df["num_episodes"].sum()

np.int64(1617)

In [19]:
unique_datasets = set(df["dataset_name"].unique())
print(unique_datasets)

{'vid2room', 'spoc', 'behavior-1k-assets'}


In [23]:
from lerobot.datasets.dataset_tools import merge_datasets, delete_episodes
import tempfile

MERGED_DATASETS_ROOT = pathlib.Path("/vision/group/vid2room/merged_rollouts")
MAX_ROLLOUTS_PER_DATASET = 100

merged_datasets = {}
for dataset_name in unique_datasets:
    df_dataset = df[(df["dataset_name"] == dataset_name) & (df["split"] == "train")].sort_values(by="num_episodes", ascending=False)
    for scene_count in tqdm([1, 5, 10, 20, 50, 100]):
        if len(df_dataset) < scene_count:
            continue

        # Take the top scene_count scenes
        df_top = df_dataset.head(scene_count)

        lr_datasets = []
        for _, row in df_top.iterrows():
            lr_dataset_name = row["lr_dataset_path"].name
            scene_ds = LeRobotDataset(root=row["lr_dataset_path"], repo_id=lr_dataset_name)

            # If there are more than the appropriate number of rollouts, take the first set
            max_rollouts = MAX_ROLLOUTS_PER_DATASET // scene_count 
            if scene_ds.num_episodes > max_rollouts:
                indices_to_delete = list(range(scene_ds.num_episodes))[max_rollouts:]
                scene_ds = delete_episodes(
                    dataset=scene_ds,
                    episode_indices=indices_to_delete,
                    output_dir=pathlib.Path(tempfile.mkdtemp(dir="/scr2/tmp")) / lr_dataset_name,
                    repo_id=lr_dataset_name,
                )

            # Add to the dataset list.
            lr_datasets.append(scene_ds)

        output_name = f"{dataset_name}-{scene_count}"
        merged = merge_datasets(lr_datasets, output_repo_id=output_name, output_dir=MERGED_DATASETS_ROOT / output_name)
        merged_datasets[output_name] = merged
        print(f"Merged {dataset_name} with {scene_count} scenes: {merged.num_episodes} episodes")
          


  0%|          | 0/6 [00:00<?, ?it/s]

Copy data and videos: 100%|██████████| 1/1 [00:00<00:00,  8.12it/s]


Merged vid2room with 1 scenes: 10 episodes


Copy data and videos: 100%|██████████| 5/5 [00:01<00:00,  3.98it/s]


Merged vid2room with 5 scenes: 50 episodes


...

Merged behavior-1k-assets with 1 scenes: 10 episodes


Copy data and videos: 100%|██████████| 5/5 [00:02<00:00,  2.49it/s]


Merged behavior-1k-assets with 5 scenes: 50 episodes


Copy data and videos: 100%|██████████| 10/10 [00:04<00:00,  2.15it/s]


Merged behavior-1k-assets with 10 scenes: 100 episodes


In [24]:
for merged_dataset_name, merged_dataset in merged_datasets.items():
    print(f"{merged_dataset_name}: {merged_dataset.num_episodes} episodes")

vid2room-1: 10 episodes
vid2room-5: 50 episodes
vid2room-10: 100 episodes
vid2room-20: 100 episodes
vid2room-50: 100 episodes
spoc-1: 10 episodes
spoc-5: 50 episodes
spoc-10: 100 episodes
spoc-20: 100 episodes
spoc-50: 100 episodes
behavior-1k-assets-1: 10 episodes
behavior-1k-assets-5: 50 episodes
behavior-1k-assets-10: 100 episodes


In [45]:
# Now, for each dataset type, load all the validation datasets,
# find 50 rollouts that are as distributed as possible across the scenes,
# and dump their episode metadatas into individual JSON files inside eval_configurations.
# These will serve as validated starting configurations for policy evaluations.
import json

TOTAL_ROLLOUTS_PER_DATASET = 50
all_rollouts = []

for dataset_name in unique_datasets:
    df_dataset = df[(df["dataset_name"] == dataset_name) & (df["split"] != "train")].sort_values(by="num_episodes", ascending=False)

    # Collect all available episodes per scene
    scenes = []
    for _, row in df_dataset.iterrows():
        available_episodes = []
        for episode_index in range(row["num_episodes"]):
            episode_metadata_path = row["lr_dataset_path"] / "meta" / f"episode_metadata_{episode_index}.json"
            if episode_metadata_path.exists():
                available_episodes.append(episode_index)
        if available_episodes:
            scenes.append({"row": row, "available": available_episodes, "selected": []})

    total_available = sum(len(s["available"]) for s in scenes)
    target = min(TOTAL_ROLLOUTS_PER_DATASET, total_available)

    # Round-robin: cycle through scenes, picking one episode at a time from each,
    # so the allocation stays as even as possible even when scenes have different counts.
    total_selected = 0
    while total_selected < target:
        for scene in scenes:
            if total_selected >= target:
                break
            if len(scene["selected"]) < len(scene["available"]):
                scene["selected"].append(scene["available"][len(scene["selected"])])
                total_selected += 1

    if total_selected < TOTAL_ROLLOUTS_PER_DATASET:
        print(f"Warning: only {total_selected} rollouts available for {dataset_name} (wanted {TOTAL_ROLLOUTS_PER_DATASET}).")

    for scene in scenes:
        if scene["selected"]:
            all_rollouts.append({
                "dataset_name": dataset_name,
                "lr_dataset_path": str(scene["row"]["lr_dataset_path"]),
                "rollout_ids": scene["selected"],
            })

with open("eval_configurations.json", "w") as f:
    json.dump(all_rollouts, f)


In [46]:
all_rollouts

[{'dataset_name': 'vid2room',
  'lr_dataset_path': '/vision/group/vid2room/rollouts/vid2room-val/vid_01bTY_glskw_office_1-lg7k19',
  'rollout_ids': [0, 1, 2, 3, 4, 5]},
 {'dataset_name': 'vid2room',
  'lr_dataset_path': '/vision/group/vid2room/rollouts/vid2room-val/vid_4l90WJiGu38_office_0-pggojq',
  'rollout_ids': [0, 1, 2, 3, 4, 5]},
 {'dataset_name': 'vid2room',
  'lr_dataset_path': '/vision/group/vid2room/rollouts/vid2room-val/vid_3LnuhhhyvDY_dining_room_0-2rb3ky',
  'rollout_ids': [0, 1, 2, 3, 4, 5]},
 {'dataset_name': 'vid2room',
  'lr_dataset_path': '/vision/group/vid2room/rollouts/vid2room-val/vid_0rbaeGo7R4k_living_room_1-avmrp9',
  'rollout_ids': [0, 1, 2, 3, 4, 5]},
 {'dataset_name': 'vid2room',
  'lr_dataset_path': '/vision/group/vid2room/rollouts/vid2room-val/vid_0n6LZYE3qGc_dining_room_1-lpc4k7',
  'rollout_ids': [0, 1, 2, 3, 4, 5]},
 {'dataset_name': 'vid2room',
  'lr_dataset_path': '/vision/group/vid2room/rollouts/vid2room-val/vid_06YiznMdE_Y_living_room_1-blu57y',
  'r